# Hospital Readmission Prediction
### L2-Regularized Logistic Regression

Predicts 30-day hospital readmission risk from patient demographics, diagnosis,
procedures, length of stay, comorbidity burden, and discharge disposition.

**Approach:** L2-regularized (ridge) logistic regression — chosen for interpretability
(clinicians need to see *why* a patient is flagged) and stability with a modest number
of features.

**Data source:** Kaggle dataset ["Hospital Readmission Prediction"](https://www.kaggle.com/datasets/vanpatangan/readmission-dataset)
(`vanpatangan/readmission-dataset`). This notebook expects the extracted archive
(`train_df.csv`, `test_df.csv`, `sample_submission.csv`) to sit in `DATA_DIR` below.

`train_df.csv` is labeled and used for model training + validation. `test_df.csv` has
no `readmitted` column — it's the Kaggle-style holdout used to produce a submission
file in the format of `sample_submission.csv`.


## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    precision_recall_curve, confusion_matrix, classification_report
)
import matplotlib.pyplot as plt

RANDOM_STATE = 42


## 2. Data

Loads the Kaggle "Hospital Readmission Prediction" files. Update `DATA_DIR` to point
at the folder where you extracted `archive.zip` (defaults to the current directory).

In [ ]:
DATA_DIR = "."  # <-- change if train_df.csv / test_df.csv live elsewhere

train_path = os.path.join(DATA_DIR, "train_df.csv")
test_path = os.path.join(DATA_DIR, "test_df.csv")
sample_submission_path = os.path.join(DATA_DIR, "sample_submission.csv")

train_df = pd.read_csv(train_path)
kaggle_test_df = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_submission_path)

print(f"train_df shape: {train_df.shape}")
print(f"test_df shape (unlabeled, for submission): {kaggle_test_df.shape}")
print(f"sample_submission shape: {sample_submission.shape}")
train_df.head()


### 2a. Inspect Data

Confirms column names/types and checks for missing values before defining feature
groups.

In [ ]:
train_df.info()
print()
print("Missing values per column:")
print(train_df.isnull().sum())
print()
print("Readmission rate (train_df):", f"{train_df['readmitted'].mean():.2%}")


### 2b. Feature / Target Configuration

Columns actually present in this dataset:
- **Numeric:** `age`, `num_procedures`, `days_in_hospital`, `comorbidity_score`
- **Categorical:** `gender` (2 levels), `primary_diagnosis` (5 levels), `discharge_to` (4 levels)
- **Target:** `readmitted` (binary, only present in `train_df`)

Categoricals are one-hot encoded inside the model pipeline (Section 4), so no manual
encoding is needed here.

In [ ]:
NUMERIC_FEATURES = ["age", "num_procedures", "days_in_hospital", "comorbidity_score"]
CATEGORICAL_FEATURES = ["gender", "primary_diagnosis", "discharge_to"]
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET_COLUMN = "readmitted"

missing = [c for c in FEATURE_COLUMNS + [TARGET_COLUMN] if c not in train_df.columns]
if missing:
    print(f"WARNING: these columns are not in train_df: {missing}")
    print(f"Available columns: {list(train_df.columns)}")
else:
    print("All expected columns found.")


## 3. Train / Validation Split

`test_df.csv` has no labels (it's the Kaggle submission set), so model evaluation uses
a stratified split carved out of `train_df.csv` instead.

In [ ]:
X = train_df[FEATURE_COLUMNS]
y = train_df[TARGET_COLUMN]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape}, Validation: {X_val.shape}")
print(f"Train readmit rate: {y_train.mean():.2%}, Validation readmit rate: {y_val.mean():.2%}")


## 4. Train Model — L2 Logistic Regression with CV-tuned C

Pipeline: one-hot encode categoricals + standardize numeric features -> L2-penalized
logistic regression with `class_weight='balanced'` to handle class imbalance
(~19% positive rate). `C` (inverse of L2 strength lambda) is tuned via 5-fold
stratified cross-validation, scored on ROC-AUC.

In [ ]:
def build_preprocessor():
    return ColumnTransformer([
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ])


def train_model(X_train, y_train):
    pipeline = Pipeline([
        ("preprocess", build_preprocessor()),
        ("clf", LogisticRegression(
            solver="lbfgs",  # default penalty is L2
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ])

    param_grid = {"clf__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10]}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    grid = GridSearchCV(
        pipeline, param_grid, scoring="roc_auc", cv=cv, n_jobs=-1
    )
    grid.fit(X_train, y_train)
    print(f"Best C (inverse of L2 strength lambda): {grid.best_params_['clf__C']}")
    print(f"Best CV ROC-AUC: {grid.best_score_:.4f}")
    return grid.best_estimator_

model = train_model(X_train, y_train)


## 5. Evaluate on Held-Out Validation Set

**Note on this dataset:** this particular Kaggle dataset appears to be randomly
generated with little to no real relationship between the features and `readmitted`
(expect ROC-AUC close to ~0.50, i.e. near chance). The pipeline below is fully correct
and will properly pick up signal if you swap in a dataset with genuine predictive
structure (e.g. the UCI "Diabetes 130-US hospitals" dataset) — don't be surprised if
metrics here look weak.

In [ ]:
def evaluate_model(model, X_eval, y_eval, label="Validation"):
    y_proba = model.predict_proba(X_eval)[:, 1]

    roc_auc = roc_auc_score(y_eval, y_proba)
    pr_auc = average_precision_score(y_eval, y_proba)
    print(f"{label} ROC-AUC: {roc_auc:.4f}")
    print(f"{label} PR-AUC : {pr_auc:.4f}")

    y_pred_default = (y_proba >= 0.5).astype(int)
    print(f"\n--- Classification report @ threshold = 0.5 ({label}) ---")
    print(classification_report(y_eval, y_pred_default, digits=3, zero_division=0))

    return y_proba, roc_auc, pr_auc

y_proba, roc_auc, pr_auc = evaluate_model(model, X_val, y_val, label="Validation")


## 6. Clinical Cost-Based Threshold Selection

The default 0.5 threshold assumes false negatives and false positives cost the same.
Clinically they don't:
- **False Negative** (missed at-risk patient): preventable readmission, patient harm,
  CMS/HRRP penalty -- high cost.
- **False Positive** (unnecessary follow-up): a phone call or care-coordination visit --
  low cost.

We sweep thresholds and pick the one minimizing `FN x cost_FN + FP x cost_FP`.

In [ ]:
def cost_based_threshold(y_eval, y_proba, cost_fn=10, cost_fp=1):
    thresholds = np.linspace(0.01, 0.99, 99)
    costs = []
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_eval, y_pred).ravel()
        total_cost = fn * cost_fn + fp * cost_fp
        costs.append(total_cost)

    best_idx = int(np.argmin(costs))
    best_threshold = thresholds[best_idx]

    print(f"Assumed cost ratio  FN:FP = {cost_fn}:{cost_fp}")
    print(f"Optimal threshold          = {best_threshold:.2f}")
    print(f"Minimum expected cost      = {costs[best_idx]:.0f}")

    y_pred_opt = (y_proba >= best_threshold).astype(int)
    print("\n--- Classification report @ cost-optimal threshold ---")
    print(classification_report(y_eval, y_pred_opt, digits=3, zero_division=0))

    return best_threshold, thresholds, costs

best_threshold, thresholds, costs = cost_based_threshold(y_val, y_proba, cost_fn=10, cost_fp=1)


## 7. Feature Importance (Interpretability)

Standardized coefficients and odds ratios -- key for clinical trust and adoption.
Feature names are pulled from the fitted `ColumnTransformer` so one-hot encoded
categories are labeled correctly.

In [ ]:
def show_feature_importance(model):
    feature_names = model.named_steps["preprocess"].get_feature_names_out()
    coefs = model.named_steps["clf"].coef_[0]
    importance = pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefs,
        "odds_ratio": np.exp(coefs),
    }).sort_values("coefficient", key=abs, ascending=False)
    print(importance.to_string(index=False))
    return importance

importance_df = show_feature_importance(model)


## 8. Plots — ROC Curve, PR Curve, Risk Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

fpr, tpr, _ = roc_curve(y_val, y_proba)
axes[0].plot(fpr, tpr, label=f"ROC-AUC = {roc_auc_score(y_val, y_proba):.3f}")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve")
axes[0].legend()

prec, rec, _ = precision_recall_curve(y_val, y_proba)
axes[1].plot(rec, prec, label=f"PR-AUC = {average_precision_score(y_val, y_proba):.3f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend()

axes[2].hist(y_proba[y_val == 0], bins=30, alpha=0.6, label="Not readmitted")
axes[2].hist(y_proba[y_val == 1], bins=30, alpha=0.6, label="Readmitted")
axes[2].set_xlabel("Predicted probability")
axes[2].set_title("Predicted Risk Distribution")
axes[2].legend()

plt.tight_layout()
plt.show()


## 9. Predict on Kaggle Test Set & Build Submission File

`test_df.csv` has no `readmitted` column, so this refits the final model on **all** of
`train_df.csv` (train + validation combined) before predicting on the held-out Kaggle
test rows, then writes a submission file matching `sample_submission.csv`'s format.

In [ ]:
final_model = train_model(X, y)  # refit on full labeled training data

X_kaggle_test = kaggle_test_df[FEATURE_COLUMNS]
kaggle_test_proba = final_model.predict_proba(X_kaggle_test)[:, 1]
kaggle_test_pred = (kaggle_test_proba >= best_threshold).astype(int)

submission = sample_submission.copy()
submission["readmitted"] = kaggle_test_pred
submission.to_csv("submission.csv", index=False)

print(f"Applied threshold: {best_threshold:.2f}")
print(f"Predicted readmission rate on test set: {kaggle_test_pred.mean():.2%}")
submission.head()


## 10. Summary

- **Data:** Kaggle "Hospital Readmission Prediction" (`vanpatangan/readmission-dataset`)
  — 5,000 labeled training rows, 2,000 unlabeled test rows, ~19% base readmission rate.
- **Model:** L2-regularized logistic regression, `C` tuned via 5-fold stratified CV on
  ROC-AUC, with one-hot encoding for `gender` / `primary_diagnosis` / `discharge_to`.
- **Class imbalance:** handled via `class_weight='balanced'` rather than resampling.
- **Metrics:** ROC-AUC (overall discrimination) and PR-AUC (minority-class performance,
  more informative under imbalance), evaluated on a held-out validation split.
- **Threshold:** chosen by minimizing clinical cost (FN >> FP), not the default 0.5 --
  favors recall/sensitivity to catch more true readmissions.
- **Interpretability:** coefficients and odds ratios (mapped back to one-hot category
  names) let clinicians validate which risk factors drive predictions.
- **Submission:** final model refit on all labeled data, predictions written to
  `submission.csv` in `sample_submission.csv`'s format.

**Next steps:** calibration curve + Brier score, comparison against gradient-boosted
trees as a performance upper bound, fairness audit across demographic subgroups
(e.g. `gender`), external validation on a held-out hospital site.
